# Лабораторная работа: регрессия

**Тема:** прогнозирование медианной цены недвижимости (MEDV) в районах Бостона.

**Ход работы:** загрузка и предобработка данных, корреляционный анализ, отбор признаков, обучение линейной регрессии, затем (доп. задания) удаление выбросов, гребневая и полиномиальная регрессии.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error

FILE = "boston.csv"
RANDOM_STATE = 42


def metrics(model, X_train, X_test, y_train, y_test, name):
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    res = {
        "Модель": name,
        "R2 train": r2_score(y_train, pred_train),
        "R2 test": r2_score(y_test, pred_test),
        "RMSE train": np.sqrt(mean_squared_error(y_train, pred_train)),
        "RMSE test": np.sqrt(mean_squared_error(y_test, pred_test)),
    }
    print(f"{name}: R2 train={res['R2 train']:.3f}, R2 test={res['R2 test']:.3f}, "
          f"RMSE train={res['RMSE train']:.3f}, RMSE test={res['RMSE test']:.3f}")
    return res

## Основная часть
### 1. Загрузка данных

In [2]:
df = pd.read_csv(FILE)
print(df.head(), "\n")

      CRIM    ZN  INDUS  CHAS    NOX  ...    TAX  PTRATIO       B  LSTAT  MEDV
0  0.00632  18.0   2.31   0.0  0.538  ...  296.0     15.3  396.90   4.98  24.0
1  0.02731   0.0   7.07   0.0  0.469  ...  242.0     17.8  396.90   9.14  21.6
2  0.02729   0.0   7.07   0.0  0.469  ...  242.0     17.8  392.83   4.03  34.7
3  0.03237   0.0   2.18   0.0  0.458  ...  222.0     18.7  394.63   2.94  33.4
4  0.06905   0.0   2.18   0.0  0.458  ...  222.0     18.7  396.90   5.33  36.2

[5 rows x 14 columns] 



### 2. Проверка типов данных

In [3]:
print("Типы данных:\n", df.dtypes, "\n")
non_numeric = df.select_dtypes(exclude="number").columns.tolist()
if non_numeric:
    df[non_numeric] = df[non_numeric].apply(pd.to_numeric, errors="coerce")
print("Все столбцы числовые:", not non_numeric, "\n")

Типы данных:
 CRIM       float64
ZN         float64
INDUS      float64
CHAS       float64
NOX        float64
RM         float64
AGE        float64
DIS        float64
RAD        float64
TAX        float64
PTRATIO    float64
B          float64
LSTAT      float64
MEDV       float64
dtype: object 

Все столбцы числовые: True 



### 3. Пропущенные значения

In [4]:
print("Пропуски по столбцам:\n", df.isna().sum(), "\n")
df = df.fillna(df.median())

Пропуски по столбцам:
 CRIM       0
ZN         0
INDUS      0
CHAS       0
NOX        0
RM         0
AGE        0
DIS        0
RAD        0
TAX        0
PTRATIO    0
B          0
LSTAT      0
MEDV       0
dtype: int64 



### 4. Корреляционная матрица

In [5]:
corr = df.corr()
print("Корреляционная матрица:\n", corr.round(2), "\n")

Корреляционная матрица:
          CRIM    ZN  INDUS  CHAS   NOX  ...   TAX  PTRATIO     B  LSTAT  MEDV
CRIM     1.00 -0.20   0.41 -0.06  0.42  ...  0.58     0.29 -0.39   0.46 -0.39
ZN      -0.20  1.00  -0.53 -0.04 -0.52  ... -0.31    -0.39  0.18  -0.41  0.36
INDUS    0.41 -0.53   1.00  0.06  0.76  ...  0.72     0.38 -0.36   0.60 -0.48
CHAS    -0.06 -0.04   0.06  1.00  0.09  ... -0.04    -0.12  0.05  -0.05  0.18
NOX      0.42 -0.52   0.76  0.09  1.00  ...  0.67     0.19 -0.38   0.59 -0.43
RM      -0.22  0.31  -0.39  0.09 -0.30  ... -0.29    -0.36  0.13  -0.61  0.70
AGE      0.35 -0.57   0.64  0.09  0.73  ...  0.51     0.26 -0.27   0.60 -0.38
DIS     -0.38  0.66  -0.71 -0.10 -0.77  ... -0.53    -0.23  0.29  -0.50  0.25
RAD      0.63 -0.31   0.60 -0.01  0.61  ...  0.91     0.46 -0.44   0.49 -0.38
TAX      0.58 -0.31   0.72 -0.04  0.67  ...  1.00     0.46 -0.44   0.54 -0.47
PTRATIO  0.29 -0.39   0.38 -0.12  0.19  ...  0.46     1.00 -0.18   0.37 -0.51
B       -0.39  0.18  -0.36  0.05 -0.38 

### 5. Тепловая карта корреляций

In [6]:
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Корреляционная матрица (Boston)")
plt.tight_layout()
plt.show()

### 6. Выбор признаков, наиболее связанных с целевым (MEDV)

In [7]:
corr_target = corr["MEDV"].drop("MEDV")
top = corr_target.abs().sort_values(ascending=False).head(5).index.tolist()
print("Топ-5 признаков по |корреляции| с MEDV:")
print(corr_target[top].round(3), "\n")

Топ-5 признаков по |корреляции| с MEDV:
LSTAT     -0.738
RM         0.695
PTRATIO   -0.508
INDUS     -0.484
TAX       -0.469
Name: MEDV, dtype: float64 



Выбраны 5 признаков с наибольшим по модулю коэффициентом корреляции с MEDV: **LSTAT** (−0,74), **RM** (0,70), **PTRATIO** (−0,51), **INDUS** (−0,48), **TAX** (−0,47).

### 7. Диаграммы рассеяния «признак – MEDV»

In [8]:
fig, axes = plt.subplots(1, len(top), figsize=(4 * len(top), 4))
for ax, col in zip(np.atleast_1d(axes), top):
    ax.scatter(df[col], df["MEDV"], alpha=0.6)
    ax.set_xlabel(col)
    ax.set_ylabel("MEDV")
    ax.set_title(f"{col} vs MEDV (r={corr.loc[col, 'MEDV']:.2f})")
plt.tight_layout()
plt.show()

### 8. Визуальная проверка связи

По диаграммам рассеяния связь с ценой прослеживается у всех выбранных признаков: у RM она прямая, у LSTAT, PTRATIO, INDUS и TAX обратная. Для LSTAT и RM зависимость заметно нелинейная. Ни один признак не исключён, поэтому для модели остаются 5 признаков (требование «не менее 3» выполнено).

### 9–10. Факторные признаки, целевая переменная, разбиение 75:25

In [9]:
features = top

X = df[features]
y = df["MEDV"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, shuffle=True)
print("Размеры выборок:", X_train.shape, X_test.shape, "\n")

Размеры выборок: (379, 5) (127, 5) 



### 11–13. Линейная регрессия, прогнозы, R² и RMSE

In [10]:
print("=== Основная часть ===")
lin = LinearRegression().fit(X_train, y_train)
results = [metrics(lin, X_train, X_test, y_train, y_test, "Линейная регрессия")]

=== Основная часть ===
Линейная регрессия: R2 train=0.691, R2 test=0.627, RMSE train=5.233, RMSE test=5.112


**Вывод.** Метрики на обучающей и тестовой выборках близки, значит, переобучения нет. При этом R^2 примерно равно 0,63–0,69, то есть линейная модель объясняет около двух третей вариации цены.

## Дополнительные задания
### 14. Boxplot целевого признака и определение выбросов

In [11]:
print("=== Дополнительные задания ===")

plt.figure(figsize=(7, 4))
sns.boxplot(x=df["MEDV"], orient="h")
plt.title("Boxplot MEDV")
plt.show()

q1, q3 = df["MEDV"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
mask_out = (df["MEDV"] < low) | (df["MEDV"] > high)
print(f"Q1={q1}, Q3={q3}, границы 'усов': [{low:.2f}, {high:.2f}]")
print("Выбросы по правилу 1.5*IQR:", sorted(df.loc[mask_out, "MEDV"].tolist()))
print("Из них значений MEDV = 50.0:", int((df["MEDV"] == 50.0).sum()), "\n")

=== Дополнительные задания ===
Q1=17.025, Q3=25.0, границы 'усов': [5.06, 36.96]
Выбросы по правилу 1.5*IQR: [5.0, 5.0, 37.0, 37.2, 37.3, 37.6, 37.9, 38.7, 39.8, 41.3, 41.7, 42.3, 42.8, 43.1, 43.5, 43.8, 44.0, 44.8, 45.4, 46.0, 46.7, 48.3, 48.5, 48.8, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0]
Из них значений MEDV = 50.0: 16 



**Выбросы.** Выбросами считаются значения MEDV вне диапазона [5,06; 36,96] (правило 1,5·IQR): всего 40 наблюдений, среди них 16 значений ровно 50,0. Эти значения выглядят как потолок в данных: реальная цена района могла быть выше, но в таблице она обрезана.

### 15. Удаление выбросов, пересоздание выборок, переобучение

In [12]:
df_clean = df[~mask_out]
print(f"Удалено строк: {int(mask_out.sum())}, осталось: {len(df_clean)}")

X_c, y_c = df_clean[features], df_clean["MEDV"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_c, y_c, test_size=0.25, random_state=RANDOM_STATE)
lin_c = LinearRegression().fit(Xc_train, yc_train)
results.append(metrics(lin_c, Xc_train, Xc_test, yc_train, yc_test,
                       "Линейная (без выбросов)"))

Удалено строк: 40, осталось: 466
Линейная (без выбросов): R2 train=0.705, R2 test=0.681, RMSE train=3.546, RMSE test=3.482


**Как изменились показатели.** R^2 на тесте вырос с 0,627 до 0,681, RMSE на тесте снизился с 5,11 до 3,48.

**О чём это говорит.** Выбросы плохо описываются линейной зависимостью и ухудшали модель. После их удаления связь признаков с ценой стала более чёткой. Следует учитывать, что часть снижения RMSE объясняется тем, что разброс цен в очищенных данных стал меньше, а сама модель теперь не описывает самые дорогие и самые дешёвые районы.

### 16. Гребневая регрессия (Ridge)

In [13]:
ridge = Ridge(alpha=1.0).fit(Xc_train, yc_train)
results.append(metrics(ridge, Xc_train, Xc_test, yc_train, yc_test,
                       "Ridge (alpha=1)"))

Ridge (alpha=1): R2 train=0.705, R2 test=0.681, RMSE train=3.546, RMSE test=3.478


**Сравнение с линейной регрессией.** Показатели практически совпадают: R^2 = 0,705 / 0,681, RMSE на тесте 3,478 против 3,482.

Гребневая регрессия добавляет L2-штраф на размер коэффициентов и помогает бороться с переобучением и мультиколлинеарностью. В нашей задаче всего 5 признаков и более 400 наблюдений, у обычной линейной регрессии нет переобучения, поэтому штраф при alpha = 1 лишь незначительно сжимает коэффициенты и на качество почти не влияет.

### 17. Полиномиальная регрессия (степень 3)

In [14]:
poly = make_pipeline(PolynomialFeatures(degree=3), LinearRegression())
poly.fit(Xc_train, yc_train)
results.append(metrics(poly, Xc_train, Xc_test, yc_train, yc_test,
                       "Полином 3-й степени"))

print("\nИтоговая таблица:")
print(pd.DataFrame(results).round(3).to_string(index=False))

Полином 3-й степени: R2 train=0.798, R2 test=0.794, RMSE train=2.939, RMSE test=2.796

Итоговая таблица:
                 Модель  R2 train  R2 test  RMSE train  RMSE test
     Линейная регрессия     0.691    0.627       5.233      5.112
Линейная (без выбросов)     0.705    0.681       3.546      3.482
        Ridge (alpha=1)     0.705    0.681       3.546      3.478
    Полином 3-й степени     0.798    0.794       2.939      2.796


## Сравнение результатов и общий вывод

| Модель | R^2   | R^2 test | RMSE train | RMSE test |
|---|-------|----------|---|---|
| Линейная регрессия | 0,691 | 0,627    | 5,233 | 5,112 |
| Линейная (без выбросов) | 0,705 | 0,681    | 3,546 | 3,482 |
| Ridge (alpha=1) | 0,705 | 0,681    | 3,546 | 3,478 |
| Полином 3-й степени | 0,798 | 0,794    | 2,939 | 2,796 |

- Удаление выбросов улучшило качество линейной модели (R² на тесте 0,627 → 0,681).
- Ridge не дал заметного выигрыша: переобучения у линейной модели нет.
- Полиномиальная регрессия 3-й степени показала лучший результат (R² на тесте 0,794, RMSE 2,80): зависимость цены от признаков (особенно LSTAT и RM) нелинейна. Разрыв между train и test мал, поэтому переобучения не наблюдается.

**Итог:** наилучшей оказалась полиномиальная модель 3-й степени, обученная на данных без выбросов.